# LSTM Architecture Validation

This notebook demonstrates the PyTorch sequence path for historical AAPL data. It performs forward passes only; no training loop, checkpointing, or performance claim is included.

## Objective and Temporal Policy

Sequences are constructed independently inside train, validation, and test partitions. This conservative policy prevents sequences from crossing split boundaries. The target date is the final observation in each lookback window, and future observations never enter the sequence.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

project_root = Path.cwd()
if not (project_root / 'ml').exists():
    project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml.data.ingestion import MarketDataIngestionService
from ml.data.yahoo import YahooFinanceProvider
from ml.neural.alignment import create_dated_sequences
from ml.neural.config import NeuralConfig, resolve_device
from ml.neural.dataset import FinancialSequenceDataset
from ml.neural.loaders import create_test_loader, create_train_loader, create_validation_loader
from ml.neural.lstm import LSTMClassifier, LSTMRegressor
from ml.supervised import build_supervised_dataset

## Load Data and Build the Supervised Dataset

In [ ]:
raw_path = project_root / 'data' / 'raw' / 'AAPL.csv'
if raw_path.exists():
    ohlcv = pd.read_csv(raw_path, parse_dates=['date'])
else:
    ohlcv = MarketDataIngestionService(YahooFinanceProvider()).ingest('AAPL', '2020-01-01', '2026-01-01')
dataset = build_supervised_dataset(ohlcv, target_type='regression', horizon=1)
config = NeuralConfig(lookback=20, input_size=len(dataset.feature_names), hidden_size=32, batch_size=32, seed=42)
print(dataset.metadata)

## Construct Isolated Sequences and Inspect Alignment

In [ ]:
def sequences_for(X, y, dates):
    return create_dated_sequences(X.to_numpy(), y.to_numpy(), dates, config.lookback)
train_sequences = sequences_for(dataset.X_train, dataset.y_train, dataset.dates_train)
validation_sequences = sequences_for(dataset.X_validation, dataset.y_validation, dataset.dates_validation)
test_sequences = sequences_for(dataset.X_test, dataset.y_test, dataset.dates_test)
for name, values in [('train', train_sequences), ('validation', validation_sequences), ('test', test_sequences)]:
    print(name, values[0].shape, values[1].shape, values[2][0], values[2][-1])

## PyTorch Datasets and DataLoaders

In [ ]:
train_data = FinancialSequenceDataset(*train_sequences)
validation_data = FinancialSequenceDataset(*validation_sequences)
test_data = FinancialSequenceDataset(*test_sequences)
train_loader = create_train_loader(train_data, config.batch_size)
validation_loader = create_validation_loader(validation_data, config.batch_size)
test_loader = create_test_loader(test_data, config.batch_size)
batch_sequences, batch_targets = next(iter(train_loader))
print(batch_sequences.shape, batch_targets.shape)

## LSTM Regression Forward Pass

In [ ]:
regressor = LSTMRegressor(config.input_size, config.hidden_size, config.num_layers, config.dropout)
regression_output = regressor(batch_sequences)
print('regression output:', regression_output.shape)

## LSTM Classification Forward Pass

In [ ]:
classifier = LSTMClassifier(config.input_size, config.hidden_size, config.num_layers, config.dropout)
classification_logits = classifier(batch_sequences)
print('classification logits:', classification_logits.shape)
print('probabilities can later use sigmoid:', torch.sigmoid(classification_logits[:3]))

## Device Selection and Scope

The device utility selects CUDA or MPS only when available and otherwise returns CPU. This notebook does not move tensors or models to an accelerator and does not train either architecture. Full training, checkpointing, early stopping, and experiment tracking belong to the next phase.

In [ ]:
device = resolve_device()
print('selected device:', device)
print('torch version:', torch.__version__)